In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
import random

import os
import pickle
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch
import copy

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver, SPOPlus, VARasNN
import functions

import copy

In [2]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data.csv")

In [3]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')

## Experimental Pipeline

For each of 10 randomly sampled 15-asset subsets of the S&P 500:

For each test month (rolling walk-forward evaluation over the last 12 months of the sample):

1. **Data preparation**: Split data into training, validation (last 6 months before test month), and test (single month) sets. Features $z_{t-1}$ consist of the last 3 months of returns (VAR(3) lag structure).

2. **Scenario matrix**: Bootstrap 1000 return scenarios from the training period to approximate the empirical return distribution for the CVaR constraint. A separate scenario matrix including the validation period is used for test-month inference.

3. **Oracle solutions**: Precompute the optimal portfolio $w^\star(y_t)$ for every training and validation instance by solving the CVaR-LP with true returns as input. These are required to compute regret during training and early stopping.

4. **Loss normalization**: Compute SPO+ and MSE scale factors on the initialized model to ensure the two loss components are comparable when combined.

5. **Model training**: For each $\gamma \in \{0, 0.25, 0.5, 0.75, 1.0\}$, train a VAR(3) model (implemented as a linear layer with trainable log_scale parameter) by minimizing the combined loss
$$\mathcal{L}_{\text{combined}} = \gamma \cdot \mathcal{L}_{\text{SPO+}} + (1 - \gamma) \cdot \mathcal{L}_{\text{MSE}}$$
using AdamW with learning rate $10^{-3}$, batch size 16, early stopping based on validation regret with patience 2 and weight decay $10^{-2}$. Model weights are restored from the epoch achieving the lowest validation regret.

6. **Test inference**: The trained model predicts returns $\hat{y}_{\text{test}}$ for the test month. The CVaR-LP is solved with $\hat{y}_{\text{test}}$ to obtain portfolio weights $\hat{w}$, and again with true returns $y_{\text{test}}$ to obtain the oracle portfolio $w^\star$. Realized return, oracle return, and regret are recorded.

7. **Checkpointing**: Results are saved to disk after every (subset, test month) combination to prevent data loss.

Results are aggregated across all test months and asset subsets to compare DFL ($\gamma > 0$) against pure MSE training ($\gamma = 0$) in terms of realized portfolio return, test regret, and prediction accuracy.

## Specify parameters

In [4]:
beta = 0.09                 # CVaR threshold (justification in thesis)

n_test_months = 12          # last year as test period

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]    # similiar to Lee et al.

n_epochs = 10               # HYPERPARAM: number of epochs
val_length = 6              # HYPERPARAM: validation length
max_lag = 3                 # HYPERPARAM: lags of VAR model

## Final Results Loop

In [ ]:
results = []

for data_seed in range(1,11):
    # prepare random return data subset
    random.seed(data_seed)
    cols_subset = random.sample(list(return_matrix.columns), 15)
    return_matrix_subset = return_matrix[cols_subset]
    # Save preprocessed return data subset
    return_matrix_subset.to_csv(f"../data/processed/return_matrix_random_seed{data_seed}.csv", index=False)
    
    X, Y = functions.create_time_series_data_with_lags(return_matrix_subset, max_lag)

    
    for test_index in tqdm(range(1,n_test_months+1), desc="test months"):

        print(f"\nDataset {data_seed}; Test Index = {test_index}\n")
        print("Preparing Data...")

        # data preparation
        X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length)
        train_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index + val_length, num_scenarios=1000, random_seed=42)
        S, N = train_scenario_loss_matrix.shape # S scenarios × N assets

        # solver
        train_solver = CVaRSolver(
            loss_matrix=train_scenario_loss_matrix,
            N=N,
            S=S,
            alpha=0.95, # CVaR confidence level
            beta=beta
        )

        # Precompute oracle solutions
        oracle_solutions = [
            train_solver.solve(c = - mu).copy()
            for mu in tqdm(Y_train, desc="Computing oracle solutions")
        ]
        oracle_tensor = torch.tensor(
            np.array(oracle_solutions),
            dtype=torch.float32
        )

        # Precompute oracle solutions for validation set
        oracle_val_solutions = [
            train_solver.solve(c=-mu).copy()
            for mu in Y_val
        ]
        oracle_val_tensor = torch.tensor(
            np.array(oracle_val_solutions),
            dtype=torch.float32
        )


        # Prepare dataset for pytorch training
        x_scaler = StandardScaler()
        X_train = x_scaler.fit_transform(X_train)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
        train_dataset = TensorDataset(
            X_train_tensor,
            Y_train_tensor,
            oracle_tensor
        )
        batch_size = 16                                                          # HYPERPARAM: Batch size
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,        # TensorDataset shuffles all tensors together
            drop_last=False
        )

        # scale validation data using the training scaler
        X_val_scaled = x_scaler.transform(X_val)
        X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
        Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

        # specify model for scale computation
        model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
        criterion = nn.MSELoss()
        # compute scale factor for the two losses to make them comparable
        torch.manual_seed(42)
        spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, train_solver, criterion)
        print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

        print("Train Models for different loss combinations:")
        
        for gamma in gamma_levels:
            
            print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

            # specify model and optimizer
            torch.manual_seed(42)
            model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=1e-3,                                                             # HYPERPARAM: Learning rate
                weight_decay=1e-2                                                    # HYPERPARAM: weight decay
            )
            criterion = nn.MSELoss()

            # Train model and retrieve training history
            history = functions.train_model(model,
                                            n_epochs,
                                            train_loader,
                                            optimizer,
                                            criterion,
                                            train_solver,
                                            spo_scale,
                                            mse_scale,
                                            gamma,
                                            X_val_tensor,
                                            Y_val_tensor,
                                            oracle_val_tensor,
                                            early_stopping_patience=2
                                            )

            # Inference
            
            # Scale test data using training scaler
            X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))

            X_test_tensor = torch.tensor(
                X_test_scaled,
                dtype=torch.float32
            )

            # Predict expected returns
            model.eval()
            with torch.no_grad():
                mu_hat_test = model(X_test_tensor)

            mu_hat_test = mu_hat_test.cpu().numpy()[0]

            test_mse = np.mean((mu_hat_test - Y_test)**2)

            # Compute portfolio weights from predicted returns with test solver
            test_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index, num_scenarios=1000, random_seed=42)
            # test solver
            test_solver = CVaRSolver(
                loss_matrix=test_scenario_loss_matrix,
                N=N,
                S=S,
                alpha=0.95, # CVaR confidence level
                beta=beta
            )

            # Compute portfolio weights using test solver (train + val scenario matrix)
            w_hat = test_solver.solve(c=-mu_hat_test).copy()
            w_oracle = test_solver.solve(c=-Y_test).copy()

            # Realized returns
            realized_return = float(Y_test @ w_hat)
            oracle_return   = float(Y_test @ w_oracle)

            # True regret: how much return we lost by using predicted rather than true returns
            test_regret = oracle_return - realized_return

            results.append({

                # id
                "run_id": f"seed{data_seed}_g{gamma}_t{test_index}_b{beta}",

                # experiment settings
                "data_seed": data_seed,
                "beta": beta,
                "gamma": gamma,
                "test_index": test_index,

                # dimensions
                "n_train": len(X_train),
                "n_assets": N,
                "n_scenarios": S,

                # normalization factors
                "spo_scale": spo_scale,
                "mse_scale": mse_scale,

                # forecasting results
                "Y_hat_test": mu_hat_test,
                "Y_test": Y_test,
                "test_mse": test_mse,

                # portfolio results
                "weights": w_hat,
                "w_oracle": w_oracle,
                "realized_return": realized_return,
                "oracle_return": oracle_return,
                "test_regret": test_regret,

                # training history
                "history": history,
                "final_spo_loss": history["spo_loss"][-1],
                "final_mse_loss": history["mse_loss"][-1],
                "final_combined_loss": history["combined_loss"][-1],
                "early_stopping_epoch": history["early_stopping_epoch"],
                "best_epoch": history["best_epoch"],
                "best_val_regret": min(history["val_regret"]),

                # model parameters
                "model_state_dict": copy.deepcopy(model.state_dict()),
            })

        print("\n---------------------------------------------------")

        # save results after every combination of data_seed and test_index
        with open("results_checkpoint_tmp.pkl", "wb") as f:
            pickle.dump(results, f)
        os.replace("results_checkpoint_tmp.pkl", "results_checkpoint.pkl")

In [25]:
len(results)

600

# Diagnostic Run

In [ ]:
# Specify parameters

beta = 0.09                 # CVaR threshold (justification in thesis)

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]

n_epochs = 10               # HYPERPARAM: number of epochs
val_length = 6              # HYPERPARAM: validation length
max_lag = 3                 # HYPERPARAM: lags of VAR model

data_seed = 1
test_index = 1              # set examplary test index

In [ ]:
results = []
diagnostics = {}  # key: gamma. Holds the per-step diag_df for each run

# prepare random return data subset
random.seed(data_seed)
cols_subset = random.sample(list(return_matrix.columns), 15)
return_matrix_subset = return_matrix[cols_subset]

X, Y = functions.create_time_series_data_with_lags(return_matrix_subset, max_lag)

print(f"\nDataset {data_seed}; Test Index = {test_index}\n")
print("Preparing Data...")

# data preparation
X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length)
train_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index + val_length, num_scenarios=1000, random_seed=42)
S, N = train_scenario_loss_matrix.shape  # S scenarios × N assets

# solver
train_solver = CVaRSolver(
    loss_matrix=train_scenario_loss_matrix,
    N=N,
    S=S,
    alpha=0.95,  # CVaR confidence level
    beta=beta
)

# Precompute oracle solutions
oracle_solutions = [
    train_solver.solve(c=-mu).copy()
    for mu in tqdm(Y_train, desc="Computing oracle solutions")
]
oracle_tensor = torch.tensor(np.array(oracle_solutions), dtype=torch.float32)

# Precompute oracle solutions for validation set
oracle_val_solutions = [
    train_solver.solve(c=-mu).copy()
    for mu in Y_val
]
oracle_val_tensor = torch.tensor(np.array(oracle_val_solutions), dtype=torch.float32)

# Prepare dataset for pytorch training
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor, oracle_tensor)

batch_size = 16  # HYPERPARAM: Batch size
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False
)

# scale validation data using the training scaler
X_val_scaled = x_scaler.transform(X_val)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

# specify model for scale computation
model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
criterion = nn.MSELoss()
torch.manual_seed(42)
spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, train_solver, criterion)
print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

print("Train Models for different loss combinations:")

for gamma in gamma_levels:

    print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

    # specify model and optimizer
    torch.manual_seed(42)
    model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,           # HYPERPARAM: Learning rate
        weight_decay=1e-2  # HYPERPARAM: weight decay
    )
    criterion = nn.MSELoss()

    # Train model with diagnostics and retrieve training history and per-step diagnostic log
    history, diag_df = functions.train_model_diagnosis(
        model,
        n_epochs,
        train_loader,
        optimizer,
        criterion,
        train_solver,
        spo_scale,
        mse_scale,
        gamma,
        X_val_tensor,
        Y_val_tensor,
        oracle_val_tensor,
        early_stopping_patience=2
    )

    diagnostics[gamma] = diag_df

    # Inference

    # Scale test data using training scaler
    X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Predict expected returns
    model.eval()
    with torch.no_grad():
        mu_hat_test = model(X_test_tensor)

    mu_hat_test = mu_hat_test.cpu().numpy()[0]
    test_mse = np.mean((mu_hat_test - Y_test) ** 2)

    # Compute portfolio weights from predicted returns with test solver
    test_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index, num_scenarios=1000, random_seed=42)
    test_solver = CVaRSolver(
        loss_matrix=test_scenario_loss_matrix,
        N=N,
        S=S,
        alpha=0.95,  # CVaR confidence level
        beta=beta
    )

    # Compute portfolio weights using test solver (train + val scenario matrix)
    w_hat = test_solver.solve(c=-mu_hat_test).copy()
    w_oracle = test_solver.solve(c=-Y_test).copy()

    # Realized returns
    realized_return = float(Y_test @ w_hat)
    oracle_return = float(Y_test @ w_oracle)

    # True regret: how much return we lost by using predicted rather than true returns
    test_regret = oracle_return - realized_return

    results.append({

        # id
        "run_id": f"seed{data_seed}_g{gamma}_t{test_index}_b{beta}",

        # experiment settings
        "data_seed": data_seed,
        "beta": beta,
        "gamma": gamma,
        "test_index": test_index,

        # dimensions
        "n_train": len(X_train),
        "n_assets": N,
        "n_scenarios": S,

        # normalization factors
        "spo_scale": spo_scale,
        "mse_scale": mse_scale,

        # forecasting results
        "Y_hat_test": mu_hat_test,
        "Y_test": Y_test,
        "test_mse": test_mse,

        # portfolio results
        "weights": w_hat,
        "w_oracle": w_oracle,
        "realized_return": realized_return,
        "oracle_return": oracle_return,
        "test_regret": test_regret,

        # training history
        "history": history,
        "final_spo_loss": history["spo_loss"][-1],
        "final_mse_loss": history["mse_loss"][-1],
        "final_combined_loss": history["combined_loss"][-1],
        "early_stopping_epoch": history["early_stopping_epoch"],
        "best_epoch": history["best_epoch"],
        "best_val_regret": min(history["val_regret"]),

        # model parameters
        "model_state_dict": copy.deepcopy(model.state_dict()),
    })

print("\n---------------------------------------------------")

# save results (final metrics)
with open("results_spo+_diagnosis_checkpoint_tmp.pkl", "wb") as f:
    pickle.dump(results, f)
os.replace("results_spo+_diagnosis_checkpoint_tmp.pkl", "results_spo+_diagnosis_checkpoint.pkl")

# save diagnostics (per-step logs) separately
with open("diagnostics_checkpoint_tmp.pkl", "wb") as f:
    pickle.dump(diagnostics, f)
os.replace("diagnostics_checkpoint_tmp.pkl", "diagnostics_checkpoint.pkl")


Dataset 1; Test Index = 1

Preparing Data...


Computing loss scales for normalization: 100%|██████████| 18/18 [00:28<00:00,  1.60s/it]


SPO+ scale: 0.153776 ; MSE  scale: 0.012291

Train Models for different loss combinations:

Gamma = 0 (weight of SPO+ loss)

Epoch 1: combined=1.035699 | val_regret=0.034285  ✓ regret improved
Epoch 2: combined=0.958223 | val_regret=0.032738  ✓ regret improved
Epoch 3: combined=0.911338 | val_regret=0.031458  ✓ regret improved
Epoch 4: combined=0.875475 | val_regret=0.031609
Epoch 5: combined=0.847420 | val_regret=0.033467
Early stopping at epoch 5. Best val regret: 0.031458

Gamma = 0.25 (weight of SPO+ loss)

Epoch 1: combined=1.029438 | val_regret=0.033946  ✓ regret improved
Epoch 2: combined=0.952971 | val_regret=0.031758  ✓ regret improved
Epoch 3: combined=0.903802 | val_regret=0.031418  ✓ regret improved
Epoch 4: combined=0.868093 | val_regret=0.032020
Epoch 5: combined=0.838321 | val_regret=0.031147  ✓ regret improved
Epoch 6: combined=0.821665 | val_regret=0.029079  ✓ regret improved
Epoch 7: combined=0.798144 | val_regret=0.026960  ✓ regret improved
Epoch 8: combined=0.778287

# Diagnostic Run (without early stopping)

In [5]:
# Specify parameters

beta = 0.09                 # CVaR threshold (justification in thesis)

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]

n_epochs = 100               # HYPERPARAM: number of epochs
val_length = 6              # HYPERPARAM: validation length
max_lag = 3                 # HYPERPARAM: lags of VAR model

data_seed = 1
test_index = 1              # set examplary test index

In [6]:
results = []
diagnostics = {}  # key: gamma. Holds the per-step diag_df for each run

# prepare random return data subset
random.seed(data_seed)
cols_subset = random.sample(list(return_matrix.columns), 15)
return_matrix_subset = return_matrix[cols_subset]

X, Y = functions.create_time_series_data_with_lags(return_matrix_subset, max_lag)

print(f"\nDataset {data_seed}; Test Index = {test_index}\n")
print("Preparing Data...")

# data preparation
X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length)
train_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index + val_length, num_scenarios=1000, random_seed=42)
S, N = train_scenario_loss_matrix.shape  # S scenarios × N assets

# solver
train_solver = CVaRSolver(
    loss_matrix=train_scenario_loss_matrix,
    N=N,
    S=S,
    alpha=0.95,  # CVaR confidence level
    beta=beta
)

# Precompute oracle solutions
oracle_solutions = [
    train_solver.solve(c=-mu).copy()
    for mu in tqdm(Y_train, desc="Computing oracle solutions")
]
oracle_tensor = torch.tensor(np.array(oracle_solutions), dtype=torch.float32)

# Precompute oracle solutions for validation set
oracle_val_solutions = [
    train_solver.solve(c=-mu).copy()
    for mu in Y_val
]
oracle_val_tensor = torch.tensor(np.array(oracle_val_solutions), dtype=torch.float32)

# Prepare dataset for pytorch training
x_scaler = StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor, oracle_tensor)

batch_size = 16  # HYPERPARAM: Batch size
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False
)

# scale validation data using the training scaler
X_val_scaled = x_scaler.transform(X_val)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

# specify model for scale computation
model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
criterion = nn.MSELoss()
torch.manual_seed(42)
spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, train_solver, criterion)
print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

print("Train Models for different loss combinations:")

for gamma in gamma_levels:

    print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

    # specify model and optimizer
    torch.manual_seed(42)
    model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,           # HYPERPARAM: Learning rate
        weight_decay=1e-2  # HYPERPARAM: weight decay
    )
    criterion = nn.MSELoss()

    # Train model with diagnostics and retrieve training history and per-step diagnostic log
    history, diag_df = functions.train_model_diagnosis(
        model,
        n_epochs,
        train_loader,
        optimizer,
        criterion,
        train_solver,
        spo_scale,
        mse_scale,
        gamma,
        X_val_tensor,
        Y_val_tensor,
        oracle_val_tensor,
        early_stopping_patience=100
    )

    diagnostics[gamma] = diag_df

    # Inference

    # Scale test data using training scaler
    X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

    # Predict expected returns
    model.eval()
    with torch.no_grad():
        mu_hat_test = model(X_test_tensor)

    mu_hat_test = mu_hat_test.cpu().numpy()[0]
    test_mse = np.mean((mu_hat_test - Y_test) ** 2)

    # Compute portfolio weights from predicted returns with test solver
    test_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index, num_scenarios=1000, random_seed=42)
    test_solver = CVaRSolver(
        loss_matrix=test_scenario_loss_matrix,
        N=N,
        S=S,
        alpha=0.95,  # CVaR confidence level
        beta=beta
    )

    # Compute portfolio weights using test solver (train + val scenario matrix)
    w_hat = test_solver.solve(c=-mu_hat_test).copy()
    w_oracle = test_solver.solve(c=-Y_test).copy()

    # Realized returns
    realized_return = float(Y_test @ w_hat)
    oracle_return = float(Y_test @ w_oracle)

    # True regret: how much return we lost by using predicted rather than true returns
    test_regret = oracle_return - realized_return

    results.append({

        # id
        "run_id": f"seed{data_seed}_g{gamma}_t{test_index}_b{beta}",

        # experiment settings
        "data_seed": data_seed,
        "beta": beta,
        "gamma": gamma,
        "test_index": test_index,

        # dimensions
        "n_train": len(X_train),
        "n_assets": N,
        "n_scenarios": S,

        # normalization factors
        "spo_scale": spo_scale,
        "mse_scale": mse_scale,

        # forecasting results
        "Y_hat_test": mu_hat_test,
        "Y_test": Y_test,
        "test_mse": test_mse,

        # portfolio results
        "weights": w_hat,
        "w_oracle": w_oracle,
        "realized_return": realized_return,
        "oracle_return": oracle_return,
        "test_regret": test_regret,

        # training history
        "history": history,
        "final_spo_loss": history["spo_loss"][-1],
        "final_mse_loss": history["mse_loss"][-1],
        "final_combined_loss": history["combined_loss"][-1],
        "early_stopping_epoch": history["early_stopping_epoch"],
        "best_epoch": history["best_epoch"],
        "best_val_regret": min(history["val_regret"]),

        # model parameters
        "model_state_dict": copy.deepcopy(model.state_dict()),
    })

print("\n---------------------------------------------------")

# save results (final metrics)
with open("results_spo+_diagnosis_checkpoint_tmp.pkl", "wb") as f:
    pickle.dump(results, f)
os.replace("results_spo+_diagnosis_checkpoint_tmp.pkl", "results_spo+_diagnosis_checkpoint.pkl")

# save diagnostics (per-step logs) separately
with open("diagnostics_checkpoint_tmp.pkl", "wb") as f:
    pickle.dump(diagnostics, f)
os.replace("diagnostics_checkpoint_tmp.pkl", "diagnostics_checkpoint.pkl")


Dataset 1; Test Index = 1

Preparing Data...


Computing loss scales for normalization: 100%|██████████| 18/18 [00:29<00:00,  1.62s/it]


SPO+ scale: 0.149528 ; MSE  scale: 0.012456

Train Models for different loss combinations:

Gamma = 0 (weight of SPO+ loss)

Epoch 1: combined=1.022026 | val_regret=0.034285  ✓ regret improved
Epoch 2: combined=0.945587 | val_regret=0.032738  ✓ regret improved
Epoch 3: combined=0.899352 | val_regret=0.031458  ✓ regret improved
Epoch 4: combined=0.864002 | val_regret=0.031607
Epoch 5: combined=0.836327 | val_regret=0.033467
Epoch 6: combined=0.821887 | val_regret=0.029525  ✓ regret improved
Epoch 7: combined=0.799078 | val_regret=0.026439  ✓ regret improved
Epoch 8: combined=0.779306 | val_regret=0.026746
Epoch 9: combined=0.785641 | val_regret=0.025721  ✓ regret improved
Epoch 10: combined=0.761056 | val_regret=0.026372
Epoch 11: combined=0.749577 | val_regret=0.025412  ✓ regret improved
Epoch 12: combined=0.733603 | val_regret=0.025518
Epoch 13: combined=0.724341 | val_regret=0.024423  ✓ regret improved
Epoch 14: combined=0.718619 | val_regret=0.028729
Epoch 15: combined=0.706913 | va